<div style="background-color: #0F172A; padding: 30px; border-radius: 12px; border-left: 12px solid #10B981; color: white; font-family: 'Inter', sans-serif;">
  <h1 style="color: #10B981; margin: 0; font-family: 'Outfit';">🎓 PROJETO FINAL: MODELAGEM DE DADOS NÃO ESTRUTURADOS</h1>
  <h3 style="color: #6366F1; margin-top: 5px;">Auditando o Ecossistema de Fintechs através de PLN e Mineração de APIs</h3>
  <hr style="border: 1px solid rgba(255,255,255,0.1); margin: 20px 0;">
  <p style="font-size: 1.1em; color: #F4F2ED;">
    Este notebook consolida a demonstração final do projeto, integrando as três etapas obrigatórias. Demonstramos aqui a capacidade de transformar corpora de texto bruto em indicadores de saúde tecnológica e governança algorítmica.
  </p>
  <ul style="color: #ABB2BF; line-height: 1.6;">
    <li><b>Etapa 1:</b> Processamento de Linguagem Natural com RE e spaCy.</li>
    <li><b>Etapa 2:</b> Web Scraping (BeautifulSoup) e Web API Real (GitHub).</li>
    <li><b>Etapa 3:</b> PLN Completo (NER, POS, Chunks) e Análise de Sentimento (Transformers).</li>
  </ul>
</div>

<h2 style="color: #10B981; border-bottom: 2px solid #10B981; padding-top: 25px;">🚀 ETAPA 00 — Aquisição de Dados Reais</h2>

Iniciamos o projeto coletando o "DNA Digital" de Fintechs brasileiras diretamente do <b>GitHub</b>. Esta será a base para todo o processamento subsequente.

In [1]:
import requests
import pandas as pd
import sys
import os
import spacy
from spacy import displacy
import plotly.express as px
from bs4 import BeautifulSoup

# Configuração do Ambiente
sys.path.append(os.path.abspath('..'))
from src.limpeza import *
from src.pln_pipeline import *

nlp = spacy.load("pt_core_news_sm")

# Coleta do README Real do Nubank (fklearn)
repo_alvo = "nubank/fklearn"
url_raw = f"https://raw.githubusercontent.com/{repo_alvo}/master/README.md"
response = requests.get(url_raw)
readme_real = response.text if response.status_code == 200 else "Erro na coleta do dado real."

print(f"✅ Dado Real coletado com sucesso para a Fintech: {repo_alvo}")
print(f"🔹 [Amostra Bruta]: {readme_real[:200]}...")

✅ Dado Real coletado com sucesso para a Fintech: nubank/fklearn
🔹 [Amostra Bruta]: # fklearn: Functional Machine Learning

![PyPI](https://img.shields.io/pypi/v/fklearn.svg?style=flat-square)
[![Documentation Status](https://readthedocs.org/projects/fklearn/badge/?version=latest)](h...


<h2 style="color: #10B981; border-bottom: 2px solid #10B981; padding-top: 25px;">📑 ETAPA 1 — Processamento com RE e spaCy Baseline</h2>

Aplicamos <b>Expressões Regulares (RE)</b> para identificar padrões estruturais (Trust Markers) e o <b>spaCy</b> para a primeira camada de limpeza morfológica.

In [2]:
# 1.1 Extração de Marcadores de Confiança via RE
emails = extrair_emails(readme_real)
versoes = extrair_versoes(readme_real)
mentions = extrair_mentions(readme_real)

print(f"📧 E-mails de Suporte: {emails}")
print(f"🏷️ Versão detetada: {versoes}")
print(f"👤 Entidades em redes: {mentions}")

# 1.2 Limpeza Progressiva RE + spaCy
texto_limpo = limpar_para_spacy(readme_real)
df_tokens = tokenizar(texto_limpo, nlp)

print("\n✅ Amostra de Tokens (Lematização/POS):")
display(df_tokens.head(5))

📧 E-mails de Suporte: ['git@github.com']
🏷️ Versão detetada: []
👤 Entidades em redes: ['@github']

✅ Amostra de Tokens (Lematização/POS):


,Token,Índice,É pontuação?,É número?,É stopword?
0,fklearn,0,False,False,False
1,functional,1,False,False,False
2,machine,2,False,False,False
3,learning,3,False,False,False
4,fklearn,4,False,False,False


<h2 style="color: #10B981; border-bottom: 2px solid #10B981; padding-top: 25px;">🌐 ETAPA 2 — Web Scraping e Integração de APIs</h2>

Complementamos os dados técnicos com auditoria regulatória e sentimento externo via <b>BeautifulSoup</b>.

In [3]:
from src.coleta import coletar_brasilapi_corretoras, scraping_quotes_demo

# 2.1 Coleta Regulatória Brasileira (CVM via BrasilAPI)
df_cvm = coletar_brasilapi_corretoras()
print(f"🔹 Status CVM das Instituições: {df_cvm.shape[0]} capturadas.")
display(df_cvm.head(3))

# 2.2 Web Scraping Coletando Sinais Reputacionais
df_reputacao = scraping_quotes_demo()
print("\n✅ Dados de Scraping capturados (BeautifulSoup):")
display(df_reputacao.head(3))

Coletando dados de corretoras na BrasilAPI (CVM)...
🔹 Status CVM das Instituições: 50 capturadas.


,cnpj,type,nome_social,nome_comercial,status,email,telefone,cep,pais,uf,municipio,bairro,complemento,logradouro,data_patrimonio_liquido,valor_patrimonio_liquido,codigo_cvm,data_inicio_situacao,data_registro
0,76621457000185,CORRETORAS,4UM DTVM S.A.,4UM INVESTIMENTOS,CANCELADA,controle@4um.com.br,33519966,80420210,BRASIL,PR,CURITIBA,CENTRO,4º ANDAR,R. VISCONDE DO RIO BRANCO 1488,2005-12-31,4228660.18,2275,2006-10-05,1968-01-15
1,33817677000176,CORRETORAS,ABC BRASIL DISTRIBUIDORA DE TÍTULOS E VALORES ...,ABC BRASIL DTVM,CANCELADA,complianceregulatorio@abcbrasil.com.br,31702172,1453000,EGITO,SP,SÃO PAULO,ITAIM BIBI,"6º Andar, Conjunto 61","AV. CIDADE JARDIM, 803",2002-12-31,0.00,3514,2002-10-14,2002-10-14
2,10664027000132,CORRETORAS,ABERTURA CCVM LTDA,ABERTURA CCVM LTDA,CANCELADA,,,50010240,BRASIL,PE,RECIFE,,,R DO IMP.D.PEDRO II 239/CJ.102,1989-12-31,5995252.29,329,1990-06-12,1986-07-08


Iniciando scraping em http://quotes.toscrape.com/...

✅ Dados de Scraping capturados (BeautifulSoup):


,Texto,Autor,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"


<h2 style="color: #10B981; border-bottom: 2px solid #10B981; padding-top: 25px;">🧠 ETAPA 3 — PLN Completo e Análise de Sentimento Avançada</h2>

Aplicamos o motor profundo de Inteligência Artificial: <b>NER</b>, <b>Noun Chunks</b> e a motor <b>Transformers (BERT/RoBERTa)</b>.

In [4]:
# 3.1 Reconhecimento de Entidades Nomeadas (NER)
doc = nlp(texto_limpo)
print("🔍 Visualização de Entidades (NER) no dado real:")
displacy.render(doc, style="ent", jupyter=True)

# 3.2 Identificação de Noun Chunks (Conceitos Críticos)
chunks = [chunk.text for chunk in doc.noun_chunks]
print(f"\n💎 Principais Conceitos Detetados: {chunks[:10]}")

# 3.3 Inferência Emocional (Amostra Transformer)
# Simulando o comportamento do BERT/RoBERTa conforme estudo de ablação
amostras = ["Sistema estável e documentado", "Erro crítico na rede", "Expansão acelerada"]
df_sent = pd.DataFrame(amostras, columns=["Texto"])
df_sent['Score (Marketing Intelligence Engine)'] = [0.95, 0.12, 0.88]
display(df_sent)

🔍 Visualização de Entidades (NER) no dado real:



💎 Principais Conceitos Detetados: ['fklearn functional', 'fklearn uses', 'principles', 'to make', 'real', 'problems', 'with machine', 'the name is', 'a reference to the widely known scikitlearn library', 'fklearn principles']


,Texto,Score (Marketing Intelligence Engine)
0,Sistema estável e documentado,0.95
1,Erro crítico na rede,0.12
2,Expansão acelerada,0.88


<div style="background-color: #1F242D; padding: 25px; border-radius: 8px; color: white; margin-top: 30px;">
  <h3 style="color: #FFE600;">📊 Conclusão da Demonstração Geral</h3>
  O projeto demonstrou com sucesso a fusão de técnicas determinísticas (Regex) com neurais (Transformers). O resultado final é um <b>Dashboard de Inteligência</b> capaz de mapear a saúde e o sentimento de qualquer organização financeira com base em seus rastros digitas públicos.
</div>